# Integrated Gas-Solar Hybrid Asset Model — Nigeria
## A Three-Equation SUR System with Correlated Monte Carlo and Project Finance

**Profile:** BSc Geology | MSc Petroleum & Energy Engineering | MSc Financial Engineering | Solar experience

---

## The Analytical Argument

To optimise a gas-solar hybrid energy asset in Nigeria, three quantities must be modelled simultaneously and jointly:

1. **Gas supply price** — what it costs to fuel a gas thermal plant
2. **Gas thermal plant capacity factor** — how much of that plant actually runs
3. **Solar PV capacity factor** — how much the solar component generates

These are not independent. They share structural drivers: FX crises raise O&M costs for both technologies. Grid instability curtails both simultaneously, if solar connects to the National grid. Sector liquidity failures reduce offtake for both. When NLNG utilisation rises, domestic gas supply tightens and power output falls (Nigerias Domestic Gas Obligation Regulation tries to solve this) — while solar is unaffected but faces the same grid evacuation bottleneck.

Modelling them independently ignores this cross-equation error correlation. This notebook estimates all three equations jointly as a **Seemingly Unrelated Regression (SUR/FGLS)** system, simulates them jointly under a correlated stochastic framework, and feeds the joint outputs into a project finance waterfall to find the Sharpe-optimal gas-solar allocation and hedge ratio.

---

## Full Notebook Flow

```
PART 1 — FOUNDATIONS
  Cell 0:  Imports and configuration
  Cell 1:  Data assembly and variable catalogue (30+ variables, 8 categories)
  Cell 2:  Stationarity testing (ADF — every variable)
  Cell 3:  Cointegration testing (Engle-Granger — Equation 1)

PART 2 — THE THREE-EQUATION REGRESSION SYSTEM
  Cell 4:  Equation 1 — Gas price (OLS + NW-HAC SE)
  Cell 5:  Full regressor universe for Eq 2 and Eq 3
  Cell 6:  Multicollinearity audit (VIF — documenting the problem)
  Cell 7:  PCA factor reduction (solving multicollinearity — 20+ → orthogonal factors)
  Cell 8:  Equation 2 — Gas thermal CF (OLS on PCA factors + NW-HAC)
  Cell 9:  Equation 3 — Solar CF (OLS on PCA factors + NW-HAC)
  Cell 10: Cross-equation residual correlation test (SUR motivation)
  Cell 11: SUR estimation via FGLS (joint efficiency gain documented)

PART 3 — CORRELATED MONTE CARLO ENGINE
  Cell 12: Market driver covariance and Cholesky decomposition
  Cell 13: Joint path simulation (gas price, gas CF, solar CF from same factors)

PART 4 — PROJECT FINANCE
  Cell 14: Stochastic dispatch and project finance waterfall
  Cell 15: Risk metrics (P10/P50/P90, VaR, CVaR, DSCR)

PART 5 — PORTFOLIO AND HEDGE OPTIMISATION
  Cell 16: Sharpe-optimal gas-solar allocation
  Cell 17: Periodic hedge ratio optimisation

PART 6 — REAL DATA INTEGRATION
  Cell 18: Data pipeline (runnable with graceful fallback)
```

---

## Colour coding
- `# ── PLACEHOLDER ──` = synthetic data; real source documented
- `# ── SUR ──` = new jointly-estimated section
- `# ── FIX ──` = corrected from v1


---
# PART 1 — FOUNDATIONS
## Cell 0 — Imports and Configuration

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 0 — IMPORTS
# pip install pandas numpy matplotlib scipy scikit-learn
# ══════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.optimize import minimize
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'#F9F9F9',
    'axes.grid':True,'grid.alpha':0.25,
    'axes.spines.top':False,'axes.spines.right':False,'font.size':10
})

START_DATE = '2015-01-01';  END_DATE  = '2025-12-31'
N_SIMS     = 5000;          N_MONTHS  = 120
RANDOM_SEED= 42;            np.random.seed(RANDOM_SEED)

print(f'Config: {N_SIMS:,} sims × {N_MONTHS} months | seed={RANDOM_SEED}')

---
## Cell 1 — Data Assembly and Variable Catalogue

Every variable has a documented source. Synthetic series are flagged `# ── PLACEHOLDER ──` and will be updated as data becomes available.

### Variable catalogue by equation

**Equation 1 — Gas price drivers:**
`ln(brent)`, `ln(usdngn)`, `ln(demand_mw)`, `ttf_jkm`, `nlng_util`

**Equation 2 — Gas thermal CF drivers (8 categories, 20 variables):**

| Category | Variables |
|---|---|
| Gas supply | gas_to_power, domestic_alloc, gas_price (from Eq1), nlng_util, ttf_jkm |
| FX crisis | ln_usdngn, fx_reserves, inflation, receivables |
| LNG exports | jkm_netback |
| Pipeline vandalism | pipeline_down, vandalism_idx, fm_dummy |
| Maintenance | forced_outage, plant_age_gas, eaf |
| Policy | grid_demand_norm, dgso_dummy |
| Transmission | wheeling_cap, freq_collapse, tcn_curtailment |
| Sector liquidity | nbet_pay_ratio, disco_collect, market_shortfall |

**Equation 3 — Solar CF drivers (6 categories, 13 variables):**

| Category | Variables |
|---|---|
| Irradiance | ghi, dni |
| Temperature/season | temp_c, harmattan, precip_mm, aerosol_od |
| Grid evacuation | tcn_curtailment*, freq_collapse*, grid_voltage_dev |
| FX | ln_usdngn* |
| Sector liquidity | nbet_pay_ratio*, disco_collect* |
| Degradation | solar_plant_age |

*Shared with Equation 2 — creates cross-equation error correlation → SUR advantageous


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1
# Load reconstructed master panel
# No synthetic fallback required
# ═══════════════════════════════════════════════════════════════════════

import os
# Safety
_RANDOM_SEED = globals().get('RANDOM_SEED', 42)
_START_DATE  = globals().get('START_DATE', '2017-01-01')
_END_DATE    = globals().get('END_DATE', '2026-05-01')

RANDOM_SEED = _RANDOM_SEED
START_DATE  = _START_DATE
END_DATE    = _END_DATE

np.random.seed(RANDOM_SEED)

dates = pd.date_range(
    START_DATE,
    END_DATE,
    freq='MS'
)

# ──────────────────────────────────────────────────────────────
# LOAD RECONSTRUCTED PANEL
# ──────────────────────────────────────────────────────────────

PANEL_PATH = "data/master_panel_reconstructed_v2.csv"

if not os.path.exists(PANEL_PATH):
    raise FileNotFoundError(
        f"{PANEL_PATH} not found. "
        "Run reconstruction engine first."
    )

df = pd.read_csv(
    PANEL_PATH,
    index_col=0,
    parse_dates=True
)

df = df.reindex(dates).ffill().bfill()

# ──────────────────────────────────────────────────────────────
# VARIABLE ALIASES FOR BACKWARD COMPATIBILITY
# ──────────────────────────────────────────────────────────────

if "eaf" not in df.columns and "PAF" in df.columns:
    df["eaf"] = df["PAF"]

if "nbet_pay_ratio" not in df.columns and "nbet_mo_pay_ratio" in df.columns:
    df["nbet_pay_ratio"] = df["nbet_mo_pay_ratio"]

if "disco_collect" not in df.columns and "disco_collect_efficiency" in df.columns:
    df["disco_collect"] = df["disco_collect_efficiency"]


# ──────────────────────────────────────────────────────────────
# CANONICAL VARIABLE NAMES
# ──────────────────────────────────────────────────────────────

rename_map = {}

if "nbet_pay_ratio" in df.columns:
    rename_map["nbet_pay_ratio"] = "nbet_ratio"

if "Total_E_Rec_by_disco" in df.columns:
    rename_map["Total_E_Rec_by_disco"] = "receivables"

df = df.rename(columns=rename_map)

# Create standalone variables expected by old notebook cells

for col in df.columns:
    globals()[col] = df[col].values

In [ ]:
df.columns

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1A — DISTRIBUTION DIAGNOSTICS
# Review distributions before any outlier treatment
# ═══════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt

# Variables likely to contain operational outliers
review_vars = [

    # DEPENDENT VARIABLES
    'gas_price',
    'gas_cf',
    'solar_cf',

    # GAS MARKET
    'brent',
    'usdngn',
    'ttf_jkm',
    'jkm_netback',
    'inflation',
    'fx_reserves',

    # GAS CF DRIVERS
    'gas_to_power',
    'domestic_alloc',
    'eaf',
    'pipeline_down',

    # CONSTRAINTS
    'Gas_constraint',
    'Transmission_constraint',
    'Distribution_constraint',

    # POWER MARKET
    'market_shortfall',
    'ATC&C',
    'nbet_ratio',
    'freq_collapse',

    # SOLAR RESOURCE
    'ghi',
    'dni',
    'temp_c',
    'precip_mm',
    'aerosol_od',
    'harmattan'
]

available_vars = [v for v in review_vars if v in df.columns]

print("Variables found:")
print(available_vars)
    
for var in available_vars:

    s = pd.to_numeric(df[var], errors='coerce').dropna()

    if len(s) < 10:
        continue

    print('\n' + '='*70)
    print(var)

    print(s.describe())

    print(f'Skewness: {s.skew():.2f}')
    print(f'Kurtosis: {s.kurtosis():.2f}')

    plt.figure(figsize=(8,4))
    plt.hist(s, bins=20)
    plt.title(f'{var} Distribution')
    plt.xlabel(var)
    plt.ylabel('Frequency')
    plt.show()

    plt.figure(figsize=(8,2))
    plt.boxplot(s, vert=False)
    plt.title(f'{var} Boxplot')
    plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# OUTLIER DIAGNOSTICS
# ═══════════════════════════════════════════════════════════════════════

review_vars = [
    'solar_cf',
    'gas_cf',
    'ghi',
    'dni',
    'temp_c',
    'precip_mm',
    'aerosol_od',
    'nbet_ratio',
    'receivables'
]

for var in review_vars:

    if var not in df.columns:
        continue

    s = pd.to_numeric(df[var], errors='coerce').dropna()

    Q1 = s.quantile(0.25)
    Q3 = s.quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((s < lower) | (s > upper)).sum()

    print(
        f"{var:<15} "
        f"Outliers={outliers:>3} "
        f"({100*outliers/len(s):.1f}%)"
    )

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1A — DESCRIPTIVE STATISTICS
# Review all variables in master dataframe
# ═══════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd

print("=" * 80)
print("DESCRIPTIVE STATISTICS")
print("=" * 80)

# Numeric columns only
num_cols = df.select_dtypes(include=[np.number]).columns

desc_stats = pd.DataFrame(index=num_cols)

desc_stats['count'] = df[num_cols].count()
desc_stats['missing'] = df[num_cols].isna().sum()
desc_stats['mean'] = df[num_cols].mean()
desc_stats['median'] = df[num_cols].median()
desc_stats['std'] = df[num_cols].std()
desc_stats['min'] = df[num_cols].min()
desc_stats['max'] = df[num_cols].max()
desc_stats['skew'] = df[num_cols].skew()
desc_stats['kurtosis'] = df[num_cols].kurt()

# Round for readability
desc_stats = desc_stats.round(4)

print(desc_stats)

# Save for thesis documentation
desc_stats.to_csv('descriptive_statistics.csv')

print("\nSaved: descriptive_statistics.csv")
print(f"Variables analysed: {len(num_cols)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1B — CORRELATION DIAGNOSTICS
# Multicollinearity screening before econometric modelling
# ═══════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Numeric variables only
num_df = df.select_dtypes(include=[np.number])

# Pearson correlation matrix
corr_matrix = num_df.corr(method='pearson')

# Save full matrix
corr_matrix.to_csv('correlation_matrix.csv')

print("=" * 80)
print("CORRELATION MATRIX SAVED")
print("=" * 80)

print(f"Variables analysed: {corr_matrix.shape[0]}")
print("Saved: correlation_matrix.csv")

# ------------------------------------------------------------
# High-correlation pairs
# ------------------------------------------------------------

threshold = 0.70

high_corr = []

for i in range(len(corr_matrix.columns)):

    for j in range(i+1, len(corr_matrix.columns)):

        corr_val = corr_matrix.iloc[i, j]

        if abs(corr_val) >= threshold:

            high_corr.append([
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_val
            ])

high_corr_df = pd.DataFrame(
    high_corr,
    columns=['Variable_1', 'Variable_2', 'Correlation']
)

high_corr_df = high_corr_df.sort_values(
    by='Correlation',
    key=lambda s: abs(s),
    ascending=False
)

print("\n" + "="*80)
print(f"HIGH CORRELATIONS (|ρ| ≥ {threshold})")
print("="*80)

print(high_corr_df)

high_corr_df.to_csv(
    'high_correlations.csv',
    index=False
)

# ------------------------------------------------------------
# Heatmap
# ------------------------------------------------------------

plt.figure(figsize=(12,10))

plt.imshow(
    corr_matrix,
    aspect='auto'
)

plt.colorbar(label='Correlation')

plt.xticks(
    range(len(corr_matrix.columns)),
    corr_matrix.columns,
    rotation=90
)

plt.yticks(
    range(len(corr_matrix.columns)),
    corr_matrix.columns
)

plt.title('Correlation Matrix')

plt.tight_layout()

plt.show()

In [ ]:
df.columns

In [ ]:
# ============================================================
# VARIANCE INFLATION FACTOR (VIF) DIAGNOSTICS
# ===========================================================
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant


def compute_vif(df, variables, model_name):

    X = df[variables].copy()

    # remove rows with missing values
    X = X.dropna()

    # add intercept
    X = add_constant(X)

    vif_df = pd.DataFrame()

    vif_df["Variable"] = X.columns

    vif_df["VIF"] = [
        variance_inflation_factor(X.values, i)
        for i in range(X.shape[1])
    ]

    print("\n" + "="*70)
    print(model_name)
    print("="*70)

    print(
        vif_df
        .sort_values("VIF", ascending=False)
        .round(2)
        .to_string(index=False)
    )

    return vif_df


# ============================================================
# GAS PRICE MODEL
# ============================================================

gas_price_vars = [
    "usdngn",
    "brent",
    "ttf_jkm",
    "jkm_netback",
    "fx_reserves"
]

compute_vif(
    df,
    gas_price_vars,
    "GAS PRICE MODEL"
)

# ===========================================================
# Receivables
# ===========================================================

receivables_vars = [
    "Total_E_Billed_by_Disco",
    "disco_collect",
    "nbet_ratio",
    "gas_cf"
]

compute_vif(
    df,
    receivables_vars, 
    "RECEIVABLES MODEL"
)


# ============================================================
# GAS CAPACITY FACTOR MODEL
# ============================================================

gas_cf_vars = [
    "eaf",
    "domestic_alloc",
    "gas_to_power",
    "pipeline_down",
    "vandalism_idx",
    "freq_collapse"
]

compute_vif(
    df,
    gas_cf_vars,
    "GAS CF MODEL"
)

# ============================================================
# EAF
# ============================================================

eaf_vars = [
    "gas_to_power",
    "vandalism_idx",
    "pipeline_down",
    "plant_age_gas",
    "freq_collapse"
]

compute_vif(
    df,
    eaf_vars,
    "EAF MODEL"
)

# ============================================================
# SOLAR CAPACITY FACTOR MODEL
# ============================================================

solar_cf_vars = [
    "dni",
    "harmattan",
    "precip_mm",
    "aerosol_od",
    "temp_c"
]

compute_vif(
    df,
    solar_cf_vars,
    "SOLAR CF MODEL"
)

---
## Cell 2 — Stationarity Testing (ADF + KPSS)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2 — STATIONARITY DIAGNOSTICS
# ADF + KPSS
# ═══════════════════════════════════════════════════════════════════════

from statsmodels.tsa.stattools import adfuller, kpss
df["ln_receivables"] = np.log(df["receivables"])
stationarity_vars = {

    # GAS PRICE MODEL
    'gas_price'             : df['gas_price'],
    'brent'                 : df['brent'],
    'usdngn'                : df['usdngn'],
    'ttf_jkm'               : df['ttf_jkm'],
    'fx_reserves'           : df['fx_reserves'],
    'jkm_netback'           : df['jkm_netback'],

    # RECEIVABLES MODEL
    'ln_receivables'        : df['ln_receivables'],
    'nbet_ratio'            : df['nbet_ratio'],
    'disco_collect'         : df['disco_collect'],
    'gas_cf'                : df['gas_cf'],
    
    # EAF MODEL
    'eaf'                   : df['eaf'],
    'pipeline_down'         : df['pipeline_down'],
    'vandalism_idx'         : df['vandalism_idx'],
    'gas_to_power'          : df['gas_to_power'],
    'plant_age_gas'         : df['plant_age_gas'],
    'freq_collapse'         : df['freq_collapse'],

     # GAS CF MODEL
    'gas_cf'                : df['gas_cf'],
    'eaf'                   : df['eaf'],
    'pipeline_down'         : df['pipeline_down'],
    'vandalism_idx'         : df['vandalism_idx'],
    'freq_collapse'         : df['freq_collapse'],    
    'domestic_alloc'        : df['domestic_alloc'],

    # SOLAR CF MODEL
    'solar_cf'              : df['solar_cf'],
    'dni'                   : df['dni'],
    'precip_mm'             : df['precip_mm'],
    'harmattan'             : df['harmattan'],
    'aerosol_od'            : df['aerosol_od'],
    'temp_c'                : df['temp_c']
}

results = []

for name, series in stationarity_vars.items():

    s = pd.Series(series).dropna()

    try:

        adf_p = adfuller(
            s,
            autolag='AIC'
        )[1]

        kpss_p = kpss(
            s,
            regression='ct',
            nlags='auto'
        )[1]

        if adf_p < 0.05 and kpss_p > 0.05:
            verdict = 'I(0)'
            action = 'LEVEL'

        elif adf_p > 0.05 and kpss_p < 0.05:
            verdict = 'I(1)'
            action = 'DIFFERENCE'

        else:
            verdict = 'Ambiguous'
            action = 'CHECK'

        results.append([
            name,
            round(adf_p,4),
            round(kpss_p,4),
            verdict,
            action
        ])

    except Exception as ex:

        results.append([
            name,
            np.nan,
            np.nan,
            f'Error: {ex}'
        ])

stationarity_results = pd.DataFrame(
    results,
    columns=[
        'Variable',
        'ADF_p',
        'KPSS_p',
        'Conclusion',
        'Action'
    ]
)
print("\nSUMMARY")
print(
    stationarity_results["Conclusion"]
    .value_counts()
)
print(stationarity_results)

stationarity_results.to_csv(
    'stationarity_results.csv',
    index=False
)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1B — CORRELATION DIAGNOSTICS
# Multicollinearity screening before econometric modelling
# ═══════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Numeric variables only
num_df = df.select_dtypes(include=[np.number])

# Pearson correlation matrix
corr_matrix = num_df.corr(method='pearson')

# Save full matrix
corr_matrix.to_csv('correlation_matrix.csv')

print("=" * 80)
print("CORRELATION MATRIX SAVED")
print("=" * 80)

print(f"Variables analysed: {corr_matrix.shape[0]}")
print("Saved: correlation_matrix.csv")

# ------------------------------------------------------------
# High-correlation pairs
# ------------------------------------------------------------

threshold = 0.70

high_corr = []

for i in range(len(corr_matrix.columns)):

    for j in range(i+1, len(corr_matrix.columns)):

        corr_val = corr_matrix.iloc[i, j]

        if abs(corr_val) >= threshold:

            high_corr.append([
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_val
            ])

high_corr_df = pd.DataFrame(
    high_corr,
    columns=['Variable_1', 'Variable_2', 'Correlation']
)

high_corr_df = high_corr_df.sort_values(
    by='Correlation',
    key=lambda s: abs(s),
    ascending=False
)

print("\n" + "="*80)
print(f"HIGH CORRELATIONS (|ρ| ≥ {threshold})")
print("="*80)

print(high_corr_df)

high_corr_df.to_csv(
    'high_correlations.csv',
    index=False
)

# ------------------------------------------------------------
# Heatmap
# ------------------------------------------------------------

plt.figure(figsize=(12,10))

plt.imshow(
    corr_matrix,
    aspect='auto'
)

plt.colorbar(label='Correlation')

plt.xticks(
    range(len(corr_matrix.columns)),
    corr_matrix.columns,
    rotation=90
)

plt.yticks(
    range(len(corr_matrix.columns)),
    corr_matrix.columns
)

plt.title('Correlation Matrix')

plt.tight_layout()

plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2B — RESOLVE AMBIGUOUS STATIONARITY RESULTS
# KPSS (LEVEL STATIONARITY)
# ═══════════════════════════════════════════════════════════════════════

from statsmodels.tsa.stattools import adfuller, kpss

ambiguous_vars = stationarity_results.loc[
    stationarity_results["Conclusion"] == "Ambiguous",
    "Variable"
].tolist()

results_2b = []

for var in ambiguous_vars:

    s = df[var].dropna()

    try:

        adf_p = adfuller(
            s,
            autolag="AIC"
        )[1]

        kpss_p = kpss(
            s,
            regression="c",
            nlags="auto"
        )[1]

        if adf_p < 0.05 and kpss_p > 0.05:

            verdict = "I(0)"
            action = "LEVEL"

        elif adf_p > 0.05 and kpss_p < 0.05:

            verdict = "I(1)"
            action = "DIFFERENCE"

        else:

            verdict = "Still Ambiguous"
            action = "VISUAL_CHECK"

        results_2b.append([
            var,
            round(adf_p, 4),
            round(kpss_p, 4),
            verdict,
            action
        ])

    except Exception as ex:

        results_2b.append([
            var,
            np.nan,
            np.nan,
            f"Error: {ex}",
            "CHECK"
        ])

cell2b_results = pd.DataFrame(
    results_2b,
    columns=[
        "Variable",
        "ADF_p",
        "KPSS_c_p",
        "Conclusion",
        "Action"
    ]
)

print("\nCELL 2B RESULTS")
print(cell2b_results)

print("\nSUMMARY")
print(cell2b_results["Conclusion"].value_counts())

cell2b_results.to_csv(
    "stationarity_results_cell2b.csv",
    index=False
)

---
# PART 2 MODELS

## Cell 3 — GAS PRICE
ARDL and Short-Run Differenced Regression models are used here as they both try to answer different questions about the gas Price model.
With ARDL we'll be testing if the Gas Price have a dynamic relationship with the selected Gas Price Variables over time (Cointegration). That is, Current effects + lagged effects + long-run equilibrium.
Short-un Differenced Regression test is the month-to-month changes in these variables explain month-to-month changes in gas price. Purely a short-run relationship check.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — ARDL BOUNDS TEST (GAS PRICE MODEL)
# FIXED ECONOMIC SPECIFICATION
# ═══════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd

from statsmodels.tsa.ardl import ARDL
from statsmodels.tsa.ardl import UECM

# --------------------------------------------------
# DATA
# --------------------------------------------------

gas_ardl_df = df[[
    "gas_price",
    "usdngn",
    "brent",
    "ttf_jkm",
    "jkm_netback",
    "fx_reserves"
]].dropna()

# --------------------------------------------------
# LOG TRANSFORMS
# --------------------------------------------------

gas_ardl_df["ln_gas_price"] = np.log(gas_ardl_df["gas_price"])

gas_ardl_df["ln_usdngn"] = np.log(
    np.clip(gas_ardl_df["usdngn"], 1e-6, None)
)

gas_ardl_df["ln_brent"] = np.log(
    np.clip(gas_ardl_df["brent"], 1e-6, None)
)

gas_ardl_df["ln_fx"] = np.log(
    np.clip(gas_ardl_df["fx_reserves"], 1e-6, None)
)

# --------------------------------------------------
# DEPENDENT VARIABLE
# --------------------------------------------------

y = gas_ardl_df["ln_gas_price"]

# --------------------------------------------------
# EXPLANATORY VARIABLES
# --------------------------------------------------

X = gas_ardl_df[
    [
        "ln_usdngn",
        "ln_brent",
        "ttf_jkm",
        "jkm_netback",
        "ln_fx"
    ]
]

# --------------------------------------------------
# FIXED ARDL(1,1,1,1,1,1)
# --------------------------------------------------

gas_ardl_model = ARDL(
    endog=y,
    lags=1,
    exog=X,
    order=1,
    trend="c"
)

gas_ardl_res = gas_ardl_model.fit()

print("=" * 70)
print("GAS PRICE ARDL RESULTS")
print("=" * 70)

print(gas_ardl_res.summary())

# --------------------------------------------------
# UECM REPRESENTATION
# --------------------------------------------------

gas_uecm = UECM.from_ardl(gas_ardl_model)

gas_uecm_res = gas_uecm.fit()

print("\n")
print("=" * 70)
print("UECM RESULTS")
print("=" * 70)

print(gas_uecm_res.summary())

# --------------------------------------------------
# BOUNDS TEST
# --------------------------------------------------

gas_bounds = gas_uecm_res.bounds_test(case=3)

print("\n")
print("=" * 70)
print("BOUNDS TEST")
print("=" * 70)

print(gas_bounds)

print("\nF-STATISTIC:")
print(gas_bounds.stat)

# --------------------------------------------------
# INTERPRETATION
# --------------------------------------------------

print("\n")
print("=" * 70)
print("INTERPRETATION")
print("=" * 70)

print("Null Hypothesis: No Cointegration")

if gas_bounds.stat > 5:
    print("\nStrong evidence of cointegration.")

elif gas_bounds.stat > 3:
    print("\nPossible cointegration. Check critical values.")

else:
    print("\nLittle evidence of cointegration.")

# --------------------------------------------------
# ERROR CORRECTION TERM
# --------------------------------------------------

if "ci_summary" in dir(gas_uecm_res):

    print("\n")
    print("=" * 70)
    print("LONG-RUN RELATIONSHIP")
    print("=" * 70)

    print(gas_uecm_res.ci_summary())

The ARDL shows (F-statistics = 0.809, upper p-value = 0.976) that there is no cointegration. And according to the AIC selection, only the 'brent' price has a statistical significance to the Gas price (judging by p-values of both current and lagged).

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 4 — GAS PRICE MODEL
# SHORT-RUN DIFFERENCED REGRESSION
# ══════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import statsmodels.api as sm

# --------------------------------------------------
# BUILD REGRESSION DATASET
# --------------------------------------------------

gp = df[
    [
        "gas_price",
        "usdngn",
        "brent",
        "ttf_jkm",
        "jkm_netback",
        "fx_reserves"
    ]
].copy()

# --------------------------------------------------
# LOG TRANSFORMS
# --------------------------------------------------

gp["ln_gas_price"] = np.log(
    np.clip(gp["gas_price"], 1e-6, None)
)

gp["ln_usdngn"] = np.log(
    np.clip(gp["usdngn"], 1e-6, None)
)

gp["ln_brent"] = np.log(
    np.clip(gp["brent"], 1e-6, None)
)

gp["ln_fx"] = np.log(
    np.clip(gp["fx_reserves"], 1e-6, None)
)

# --------------------------------------------------
# FIRST DIFFERENCES
# --------------------------------------------------

gp["d_ln_gas_price"] = gp["ln_gas_price"].diff()

gp["d_ln_usdngn"] = gp["ln_usdngn"].diff()

gp["d_ln_brent"] = gp["ln_brent"].diff()

gp["d_ln_fx"] = gp["ln_fx"].diff()

gp["d_ttf_jkm"] = gp["ttf_jkm"].diff()

gp["d_jkm_netback"] = gp["jkm_netback"].diff()

gp = gp.dropna()

# --------------------------------------------------
# DEPENDENT VARIABLE
# --------------------------------------------------

y = gp["d_ln_gas_price"]

# --------------------------------------------------
# EXPLANATORY VARIABLES
# --------------------------------------------------

X = gp[
    [
        "d_ln_usdngn",
        "d_ln_brent",
        "d_ttf_jkm",
        "d_jkm_netback",
        "d_ln_fx"
    ]
]

X = sm.add_constant(X)

# --------------------------------------------------
# OLS + NEWEY-WEST HAC
# --------------------------------------------------

gas_model = sm.OLS(y, X)

gas_results = gas_model.fit(
    cov_type="HAC",
    cov_kwds={"maxlags":4}
)

# --------------------------------------------------
# OUTPUT
# --------------------------------------------------

print("=" * 80)
print("GAS PRICE MODEL")
print("SHORT-RUN DIFFERENCED REGRESSION")
print("=" * 80)

print(gas_results.summary())

# --------------------------------------------------
# DIAGNOSTICS
# --------------------------------------------------

print("\n")
print("=" * 80)
print("KEY METRICS")
print("=" * 80)

print(f"Observations : {int(gas_results.nobs)}")
print(f"R²           : {gas_results.rsquared:.4f}")
print(f"Adj. R²      : {gas_results.rsquared_adj:.4f}")
print(f"AIC          : {gas_results.aic:.4f}")
print(f"BIC          : {gas_results.bic:.4f}")

# --------------------------------------------------
# ECONOMIC INTERPRETATION TABLE
# --------------------------------------------------

gas_coef_table = pd.DataFrame({
    "Coefficient": gas_results.params,
    "Std_Error": gas_results.bse,
    "t_stat": gas_results.tvalues,
    "p_value": gas_results.pvalues
})

print("\n")
print("=" * 80)
print("COEFFICIENTS")
print("=" * 80)

print(gas_coef_table.round(4))

The short-run Regression analysis says that none of the candidate variables significantly explains changes in gas price. This suggests that Nigerian Domestic Gas Price is not primarily driven by Brent, FX, LNG Prices, Netback Prices and Reserves over the sample period.

In [ ]:
# ==========================================================
# LOCK FINAL GAS PRICE MODEL
# ==========================================================

gas_final = gas_results

print("Gas Price model locked")

Bounds testing found no evidence of cointegration between domestic gas prices and the selected market variables. Furthermore, short-run differenced regressions showed no statistically significant effects from changes in Brent, exchange rates, LNG benchmarks, netback prices, or foreign reserves. This suggests that domestic gas prices in Nigeria are influenced more by policy, contractual arrangements, and administrative mechanisms than by market fundamentals.

---
## EAF MODEL

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# EAF DIAGNOSTICS
# Descriptive Statistics + Time Series Plot + Correlation Matrix
# ═══════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------------------------------------
# VARIABLES
# -------------------------------------------------------------

eaf_vars = [
    'eaf',
    'pipeline_down',
    'vandalism_idx',
    'plant_age_gas',
    'gas_to_power',
    'freq_collapse'
]

eaf_df = df[eaf_vars].copy()

# -------------------------------------------------------------
# 1. EAF DESCRIPTIVE STATISTICS
# -------------------------------------------------------------

print("\n" + "="*70)
print("EAF DESCRIPTIVE STATISTICS")
print("="*70)

print(f"Mean : {eaf_df['eaf'].mean():.4f}")
print(f"Min  : {eaf_df['eaf'].min():.4f}")
print(f"Max  : {eaf_df['eaf'].max():.4f}")
print(f"Std  : {eaf_df['eaf'].std():.4f}")

print("\nFull Summary")
print(eaf_df['eaf'].describe())

# -------------------------------------------------------------
# 2. EAF TIME SERIES PLOT
# -------------------------------------------------------------

plt.figure(figsize=(12,5))

plt.plot(
    df.index,
    eaf_df['eaf'],
    linewidth=2
)

plt.title("Equivalent Availability Factor (EAF)")
plt.xlabel("Date")
plt.ylabel("EAF")

plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# -------------------------------------------------------------
# 3. EAF CORRELATION MATRIX
# -------------------------------------------------------------

eaf_corr_matrix = eaf_df.corr()

print("\n" + "="*70)
print("EAF CORRELATION MATRIX")
print("="*70)

print(eaf_corr_matrix.round(3))

plt.figure(figsize=(8,6))

sns.heatmap(
    eaf_corr_matrix,
    annot=True,
    cmap='coolwarm',
    center=0,
    fmt='.2f'
)

plt.title("EAF Model Correlation Matrix")

plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — EAF ARDL LAG SELECTION
# ═══════════════════════════════════════════════════════════════════════

from statsmodels.tsa.ardl import ardl_select_order

# -------------------------------------------------------------
# DEPENDENT VARIABLE
# -------------------------------------------------------------

y = df["eaf"]

# -------------------------------------------------------------
# EXPLANATORY VARIABLES
# -------------------------------------------------------------

X = df[
    [
        "pipeline_down",
        "vandalism_idx",
        "plant_age_gas",
        "gas_to_power",
        "freq_collapse"
    ]
]

# -------------------------------------------------------------
# AUTOMATIC ARDL ORDER SELECTION
# -------------------------------------------------------------

eaf_sel = ardl_select_order(
    endog=y,
    maxlag=4,
    exog=X,
    maxorder=4,
    ic="aic",
    trend="c"
)

print("\n" + "="*70)
print("SELECTED ARDL ORDER")
print("="*70)

print(eaf_sel.model.ardl_order)

# -------------------------------------------------------------
# FIT SELECTED MODEL
# -------------------------------------------------------------

eaf_ardl_model = eaf_sel.model.fit()

print("\n" + "="*70)
print("ARDL SUMMARY")
print("="*70)

print(eaf_ardl_model.summary())

From the Automactic ARDL Selection, variables selected are eaf.L1 (highly persistent), plant_age_gas.L0, plant_age_gas.L1, plant_age_gas.L2, freq_collapse.L0, freq_collapse.L1. The positive sign in Frequeny collapse is counterintuitive, possible reverse causality or ommitted variable effects.
The full EAF ARDL MODEL doesn't perform better than the AIC-selceted specification above, i'll keep the later.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Diagnosis
# EAF ARDL (AGE REMOVED, TREND INCLUDED)
# ═══════════════════════════════════════════════════════════════════════
y = df["eaf"]

X = df[
    [
        "pipeline_down",
        "vandalism_idx",
        "gas_to_power",
        "freq_collapse"
    ]
]

eaf_sel2 = ardl_select_order(
    endog=y,
    exog=X,
    maxlag=4,
    maxorder=4,
    ic="aic",
    trend="ct"
)

print("\nSELECTED ORDER")
print(eaf_sel2.model.ardl_order)

print("\nAR LAGS")
print(eaf_sel2.model.ar_lags)

print("\nDL LAGS")
print(eaf_sel2.model.dl_lags)

eaf_model_2 = eaf_sel2.model.fit()

print(eaf_model_2.summary())

In this alternative selection, only freq_collapse is significant (P-value < 0.05). But the model Quality, represented by AIC is much worse (683) than the previous model (631). so Previous model trumps.

In [ ]:
# ==========================================================
# FORCED ECM-COMPATIBLE EAF ARDL
# ==========================================================

from statsmodels.tsa.ardl import ARDL

eaf_theory_ardl = ARDL(
    endog=df["eaf"],
    lags=1,
    exog=df[
        [
            "pipeline_down",
            "vandalism_idx",
            "gas_to_power",
            "freq_collapse"
        ]
    ],
    order=1,
    trend="ct"
).fit()

print(eaf_theory_ardl.summary())

For theory validation, i forced in pipeline_down, vandalism_idx, gas_to_power and freq_collapse To test if the conceptual engineering stroy survives econometric scrutiny. It didn't as all the forced in variable shows not to be significant.

Since ADRL alone cannot tell us whether a long-run relationship exists, Up next, I'll run UECM test, to check if deviations from equilibrium gets corrected.

In [ ]:
from statsmodels.tsa.ardl import UECM

eaf_uecm = UECM.from_ardl(eaf_sel.model)

eaf_uecm_res = eaf_uecm.fit()

print(eaf_uecm_res.summary())

In [ ]:
forced_bt = eaf_uecm_res.bounds_test(case=5)

print(forced_bt)

The error correction term, eaf.L1 = -0.214 and p=0.000, shows that approximately 18% of disequilibrium is corrected each month. And only freq_collapse.L1 is clearly significant (P=0.000), Pipeline down is borderline.

In [ ]:
# ==========================================================
# FINAL EAF MODEL
# ==========================================================

eaf_final_model = eaf_ardl_model

Pipeline downtime, vandalism incidents, and gas-to-power allocation were initially included based on engineering considerations. However, when forced into the ARDL specification, these variables remained statistically insignificant and materially worsened model fit (higher AIC). Consequently, the final EAF specification retained only the statistically supported variables identified through ARDL lag selection.

---
## GAS CF MODEL

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# GAS CF MODEL DIAGNOSTICS
# Descriptive Statistics + Time Series Plot + Correlation Matrix
# ═══════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------------------------------------
# VARIABLES
# -------------------------------------------------------------

gas_cf_vars = [
    'gas_cf',
    'eaf',
    'pipeline_down',
    'vandalism_idx',
    'domestic_alloc',
    'freq_collapse'
]

gas_cf_df = df[gas_cf_vars].copy()

# -------------------------------------------------------------
# 1. GAS CF DESCRIPTIVE STATISTICS
# -------------------------------------------------------------

print("\n" + "="*70)
print("GAS CF DESCRIPTIVE STATISTICS")
print("="*70)

print(f"Mean : {gas_cf_df['gas_cf'].mean():.4f}")
print(f"Min  : {gas_cf_df['gas_cf'].min():.4f}")
print(f"Max  : {gas_cf_df['gas_cf'].max():.4f}")
print(f"Std  : {gas_cf_df['gas_cf'].std():.4f}")

print("\nFull Summary")
print(gas_cf_df['gas_cf'].describe())

# -------------------------------------------------------------
# 2. GAS CF TIME SERIES PLOT
# -------------------------------------------------------------

plt.figure(figsize=(12,5))

plt.plot(
    df.index,
    gas_cf_df['gas_cf'],
    linewidth=2
)

plt.title("Gas Capacity Factor (GAS CF)")
plt.xlabel("Date")
plt.ylabel("GAS CF")

plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# -------------------------------------------------------------
# 3. GAS CF CORRELATION MATRIX
# -------------------------------------------------------------

gas_corr_matrix = gas_cf_df.corr()

print("\n" + "="*70)
print("GAS CF CORRELATION MATRIX")
print("="*70)

print(gas_corr_matrix.round(3))

plt.figure(figsize=(8,6))

sns.heatmap(
    gas_corr_matrix,
    annot=True,
    cmap='coolwarm',
    center=0,
    fmt='.2f'
)

plt.title("GAS CF Model Correlation Matrix")

plt.tight_layout()
plt.show()

Next, we go through Automatic ARDL Selection for the purpose of determining the best dynamic structure.

In [111]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — GAS CF ARDL LAG SELECTION
# ═══════════════════════════════════════════════════════════════════════

from statsmodels.tsa.ardl import ardl_select_order

# -------------------------------------------------------------
# DEPENDENT VARIABLE
# -------------------------------------------------------------

y = df["gas_cf"]

# -------------------------------------------------------------
# EXPLANATORY VARIABLES
# -------------------------------------------------------------

X = df[
    [
        "eaf",
        "pipeline_down",
        "freq_collapse",
        "domestic_alloc"
    ]
]

# -------------------------------------------------------------
# AUTOMATIC ARDL ORDER SELECTION
# -------------------------------------------------------------

sel_gcf = ardl_select_order(
    endog=y,
    maxlag=4,
    exog=X,
    maxorder=4,
    ic="aic",
    trend="c"
)

print("\n" + "="*70)
print("SELECTED ARDL ORDER")
print("="*70)

print(sel_gcf.model.ardl_order)

print("\nAR LAGS")
print(sel_gcf.model.ar_lags)

print("\nDL LAGS")
print(sel_gcf.model.dl_lags)

# -------------------------------------------------------------
# FIT SELECTED MODEL
# -------------------------------------------------------------

gcf_model = sel_gcf.model.fit()

print("\n" + "="*70)
print("ARDL SUMMARY")
print("="*70)

print(gcf_model.summary())


SELECTED ARDL ORDER
(3, 1, 0, 1)

AR LAGS
[1, 2, 3]

DL LAGS
{'pipeline_down': [0, 1], 'freq_collapse': [0], 'domestic_alloc': [0, 1]}

ARDL SUMMARY
                              ARDL Model Results                              
Dep. Variable:                 gas_cf   No. Observations:                  132
Model:               ARDL(3, 1, 0, 1)   Log Likelihood                -239.680
Method:               Conditional MLE   S.D. of innovations              1.551
Date:                Tue, 09 Jun 2026   AIC                            499.360
Time:                        17:26:47   BIC                            527.958
Sample:                    04-01-2015   HQIC                           510.980
                         - 12-01-2025                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                10.9039      2.406      4

resulting structure is ARDL(3,1,0,1) and Selected variables are: gas_cf, freq_collapse, domestic_alloc, pipeline_down. with gas_cf.L1 being of dominant effect. Basically the current gas capacity factor is heavily influneced by previous gas capacity factor.
Pipeline_down current effect is weak and Lagged effect is borderline significant.
Freq_collapse is a very strong driver.

The optimal ARDL specification selected by AIC included a contemporaneous-only frequency collapse term, preventing direct UECM transformation.

In [119]:
from statsmodels.tsa.vector_ar.vecm import coint_johansen

johansen_data = df[
    [
        "gas_cf",
        "pipeline_down",
        "freq_collapse",
        "domestic_alloc"
    ]
].dropna()

jres = coint_johansen(
    johansen_data,
    det_order=0,
    k_ar_diff=1
)

print("Trace Statistics")
print(jres.lr1)

print("\nCritical Values (95%)")
print(jres.cvt[:,1])

Trace Statistics
[103.13793915  48.89815211   7.94752452   0.10612274]

Critical Values (95%)
[47.8545 29.7961 15.4943  3.8415]


Johansen cointegration testing was conducted on the final model variables. The trace statistics exceeded the 95% critical values for the first two ranks, indicating the presence of two cointegrating relationships among Gas CF, Pipeline Down, Frequency Collapse, and Domestic Allocation.

In [113]:
# ==========================================================
# LOCK GAS CF CANDIDATE MODEL
# ==========================================================

gcf_final_model = gcf_model

print(gcf_final_model.summary())

                              ARDL Model Results                              
Dep. Variable:                 gas_cf   No. Observations:                  132
Model:               ARDL(3, 1, 0, 1)   Log Likelihood                -239.680
Method:               Conditional MLE   S.D. of innovations              1.551
Date:                Tue, 09 Jun 2026   AIC                            499.360
Time:                        17:35:48   BIC                            527.958
Sample:                    04-01-2015   HQIC                           510.980
                         - 12-01-2025                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                10.9039      2.406      4.531      0.000       6.140      15.668
gas_cf.L1             0.7704      0.088      8.746      0.000       0.596       0.945
gas_cf.L2            -0.

---
## Receivables Model

In [ ]:
df['receivables'].describe()

In [ ]:
df["ln_receivables"] = np.log(df["receivables"])

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — RECEIVABLES CORRELATION ANALYSIS
# Pearson + Spearman
# ═══════════════════════════════════════════════════════════════════════

recv_vars = [
    'ln_receivables',
    'nbet_ratio',
    'disco_collect',
    'gas_cf'
]

recv_df = df[recv_vars].copy()

print("="*70)
print("PEARSON CORRELATION")
print("="*70)

pearson_corr = recv_df.corr(method='pearson')
print(pearson_corr)

print("\n")

print("="*70)
print("SPEARMAN CORRELATION")
print("="*70)

spearman_corr = recv_df.corr(method='spearman')
print(spearman_corr)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4A — ARDL LAG SELECTION
# ln_receivables
# ═══════════════════════════════════════════════════════════════════════

from statsmodels.tsa.ardl import ardl_select_order

y = df["ln_receivables"]

X = df[
    [
        "nbet_ratio",
        "disco_collect",
        "gas_cf"
    ]
]

sel_recv = ardl_select_order(
    endog=y,
    exog=X,
    maxlag=4,
    maxorder=4,
    ic="aic",
    trend="c"
)

print("\nSELECTED ORDER")
print(sel_recv.model.ardl_order)

print("\nAR LAGS")
print(sel_recv.model.ar_lags)

print("\nDL LAGS")
print(sel_recv.model.dl_lags)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4B — ARDL ESTIMATION
# ═══════════════════════════════════════════════════════════════════════

recv_ardl = sel_recv.model.fit()

print("\n" + "="*70)
print("ARDL SUMMARY")
print("="*70)

print(recv_ardl.summary())

In [ ]:
# ═══════════════════════════════════════════════════════════════
# RECEIVABLES ECM-COMPATIBLE ARDL
# ═══════════════════════════════════════════════════════════════

from statsmodels.tsa.ardl import ARDL

recv_ardl_ecm = ARDL(
    endog=df["ln_receivables"],
    lags=1,
    exog=df[
        [
            "nbet_ratio",
            "disco_collect"
        ]
    ],
    order=1,
    trend="c"
).fit()

print(recv_ardl_ecm.summary())

In [ ]:
from statsmodels.tsa.ardl import UECM

recv_uecm = UECM.from_ardl(recv_ardl_ecm.model)

recv_uecm_res = recv_uecm.fit()

print(recv_uecm_res.summary())

In [ ]:
recv_bt = recv_uecm_res.bounds_test(case=3)

print(recv_bt)

In [ ]:
recv_final = recv_ardl_ecm

---
## Solar CF

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# SOLAR CF MODEL DIAGNOSTICS
# Descriptive Statistics + Time Series Plot + Correlation Matrix
# ═══════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------------------------------------
# VARIABLES
# -------------------------------------------------------------

solar_cf_vars = [
    'solar_cf',
    'dni',
    'precip_mm',
    'harmattan',
    'aerosol_od',
    'temp_c'
]

solar_cf_df = df[solar_cf_vars].copy()

# -------------------------------------------------------------
# 1. SOLAR CF DESCRIPTIVE STATISTICS
# -------------------------------------------------------------

print("\n" + "="*70)
print("SOLAR CF DESCRIPTIVE STATISTICS")
print("="*70)

print(f"Mean : {solar_cf_df['solar_cf'].mean():.4f}")
print(f"Min  : {solar_cf_df['solar_cf'].min():.4f}")
print(f"Max  : {solar_cf_df['solar_cf'].max():.4f}")
print(f"Std  : {solar_cf_df['solar_cf'].std():.4f}")

print("\nFull Summary")
print(solar_cf_df['solar_cf'].describe())

# -------------------------------------------------------------
# 2. SOLAR CF TIME SERIES PLOT
# -------------------------------------------------------------

plt.figure(figsize=(12,5))

plt.plot(
    df.index,
    solar_cf_df['solar_cf'],
    linewidth=2
)

plt.title("Solar Capacity Factor (SOLAR CF)")
plt.xlabel("Date")
plt.ylabel("SOLAR CF")

plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# -------------------------------------------------------------
# 3. SOLAR CF CORRELATION MATRIX
# -------------------------------------------------------------
print("="*70)
print("PEARSON CORRELATION")
print("="*70)

pearson_corr = solar_cf_df.corr(method='pearson')
print(pearson_corr)

print("\n")

print("="*70)
print("SPEARMAN CORRELATION")
print("="*70)

spearman_corr = solar_cf_df.corr(method='spearman')
print(spearman_corr)

corr_matrix = solar_cf_df.corr()

print("\n" + "="*70)
print("SOLAR CF CORRELATION MATRIX")
print("="*70)

print(corr_matrix.round(3))

plt.figure(figsize=(8,6))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    center=0,
    fmt='.2f'
)

plt.title("SOLAR CF Model Correlation Matrix")

plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — SCATTER PLOTS
# Solar CF vs Drivers
# ═══════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt

solar_vars = [
    'dni',
    'precip_mm',
    'harmattan',
    'aerosol_od',
    'temp_c'
]

for var in solar_vars:

    plt.figure(figsize=(6,4))

    plt.scatter(
        df[var],
        df['solar_cf']
    )

    plt.xlabel(var)
    plt.ylabel('solar_cf')
    plt.title(f'Solar CF vs {var}')

    plt.grid(True)

    plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4A — ARDL LAG SELECTION
# ln_receivables
# ═══════════════════════════════════════════════════════════════════════

from statsmodels.tsa.ardl import ardl_select_order

y = df["solar_cf"]

X = df[
    [
        "dni",
        "precip_mm",
        "harmattan",
        "aerosol_od",
        "temp_c"
    ]
]

sel_solar = ardl_select_order(
    endog=y,
    exog=X,
    maxlag=4,
    maxorder=4,
    ic="aic",
    trend="c"
)

print("\nSELECTED ORDER")
print(sel_solar.model.ardl_order)

print("\nAR LAGS")
print(sel_solar.model.ar_lags)

print("\nDL LAGS")
print(sel_solar.model.dl_lags)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# MONTHLY SEASONALITY
# ═══════════════════════════════════════════════════════════════════════

df['month'] = df.index.month

monthly_cf = (
    df.groupby('month')['solar_cf']
      .mean()
)

print(monthly_cf)

monthly_cf.plot(
    marker='o',
    figsize=(10,5),
    title='Average Solar CF by Month'
)

plt.ylabel('Solar CF')
plt.grid(True)

plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# QUARTERLY SEASONALITY
# ═══════════════════════════════════════════════════════════════════════

df['quarter'] = df.index.quarter

quarterly_cf = (
    df.groupby('quarter')['solar_cf']
      .mean()
)

print(quarterly_cf)

quarterly_cf.plot(
    kind='bar',
    figsize=(8,5),
    title='Average Solar CF by Quarter'
)

plt.ylabel('Solar CF')

plt.grid(True)

plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# ANOVA TEST FOR MONTHLY SEASONALITY
# ═══════════════════════════════════════════════════════════════════════

from scipy.stats import f_oneway

monthly_groups = [
    group['solar_cf'].values
    for _, group in df.groupby(df.index.month)
]

f_stat, p_value = f_oneway(*monthly_groups)

print("\nANOVA RESULTS")
print("F-stat :", round(f_stat,4))
print("p-value:", round(p_value,6))

In [ ]:
solar_ardl = sel_solar.model.fit()

print(solar_ardl.summary())

In [ ]:
from statsmodels.tsa.ardl import UECM

solar_uecm = UECM.from_ardl(sel_solar.model)

solar_uecm_res = solar_uecm.fit()

print(solar_uecm_res.summary())

In [ ]:
solar_bt = solar_uecm_res.bounds_test(case=3)

print(solar_bt)

In [ ]:
month_dummies = pd.get_dummies(
    df.index.month,
    prefix="m",
    drop_first=True
)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MONTHLY DUMMY BENCHMARK MODEL
# ═══════════════════════════════════════════════════════════════

import statsmodels.api as sm
import pandas as pd

month_dummies = pd.get_dummies(
    df.index.month,
    prefix='m',
    drop_first=True,
    dtype=float
)

# IMPORTANT
month_dummies.index = df.index

X = sm.add_constant(month_dummies)

seasonal_model = sm.OLS(
    df["solar_cf"],
    X
).fit()

print(seasonal_model.summary())

In [ ]:
print(solar_uecm_res.test_heteroskedasticity())

print(solar_uecm_res.test_normality())

In [ ]:
solar_final = solar_uecm_res

In [ ]:
final_models = {
    "Gas Price": gas_final,
    "EAF": eaf_final_model,
    "Gas CF": gcf_final_model,
    "Solar CF": solar_final,
    "Receivables": recv_final
}

for name in final_models:
    print(name, "✓")

---
## Forecasting Errors

In [ ]:
# ==========================================================
# RESIDUAL EXTRACTION
# ==========================================================

gas_resid   = gas_final.resid
eaf_resid   = eaf_final_model.resid
gcf_resid   = gcf_ardl_ecm.resid
solar_resid = solar_ardl.resid
recv_resid  = recv_final.resid

print("Residuals extracted")

In [ ]:
# ==========================================================
# RESIDUAL LENGTHS
# ==========================================================

print("Gas Price   :", len(gas_resid))
print("EAF         :", len(eaf_resid))
print("Gas CF      :", len(gcf_resid))
print("Solar CF    :", len(solar_resid))
print("Receivables :", len(recv_resid))

In [ ]:
# ==========================================================
# MASTER RESIDUAL DATASET
# ==========================================================

residuals = pd.concat(
    [
        gas_resid.rename("gas_price"),
        eaf_resid.rename("eaf"),
        gcf_resid.rename("gas_cf"),
        solar_resid.rename("solar_cf"),
        recv_resid.rename("receivables")
    ],
    axis=1
)

residuals = residuals.dropna()

print(residuals.shape)

residuals.head()

In [ ]:
# ==========================================================
# PEARSON CORRELATION
# ==========================================================

pearson_resid = residuals.corr()

print(pearson_resid)

plt.figure(figsize=(8,6))
sns.heatmap(
    pearson_resid,
    annot=True,
    cmap="coolwarm",
    center=0
)

plt.title("Residual Pearson Correlation Matrix")
plt.show()

In [ ]:
# ==========================================================
# SPEARMAN CORRELATION
# ==========================================================

spearman_resid = residuals.corr(
    method="spearman"
)

print(spearman_resid)

plt.figure(figsize=(8,6))
sns.heatmap(
    spearman_resid,
    annot=True,
    cmap="coolwarm",
    center=0
)

plt.title("Residual Spearman Correlation Matrix")
plt.show()

In [ ]:
print(eaf_resid.sample(20))

print(gcf_resid.sample(20))

print(solar_resid.sample(20))

Up next, i'll use a copula, not because the residual dependence is strong but because it can capture tail co-movement even when linear correlations are small (as shown in above). But before fitting a copula, we need to understand the probability distribution of each model's forecast error.
*Residual Distribution Analysis* answers the question, what distribution best describes each residual series?

In [ ]:
# ==========================================================
# RESIDUAL SUMMARY STATISTICS
# ==========================================================

print(
    residuals.describe().T
)

In [ ]:
# ==========================================================
# SKEWNESS & KURTOSIS
# ==========================================================

from scipy.stats import skew, kurtosis

for col in residuals.columns:

    print("\n" + "="*60)
    print(col)
    print("="*60)

    print(
        "Skewness :",
        round(skew(residuals[col]),4)
    )

    print(
        "Kurtosis :",
        round(
            kurtosis(
                residuals[col],
                fisher=True
            ),
            4
        )
    )

In [ ]:
# ==========================================================
# JARQUE-BERA TEST
# ==========================================================

from scipy.stats import jarque_bera

for col in residuals.columns:

    jb = jarque_bera(
        residuals[col]
    )

    print("\n" + "="*60)
    print(col)
    print("="*60)

    print(
        "JB Statistic:",
        round(jb.statistic,4)
    )

    print(
        "p-value:",
        round(jb.pvalue,6)
    )

In [ ]:
# ==========================================================
# HISTOGRAMS
# ==========================================================

import matplotlib.pyplot as plt

for col in residuals.columns:

    plt.figure(figsize=(6,4))

    residuals[col].hist(
        bins=20
    )

    plt.title(col)

    plt.grid(True)

    plt.show()

In [ ]:
# ==========================================================
# QQ PLOTS
# ==========================================================

import scipy.stats as stats
import matplotlib.pyplot as plt

for col in residuals.columns:

    plt.figure(figsize=(6,5))

    stats.probplot(
        residuals[col],
        dist="norm",
        plot=plt
    )

    plt.title(
        f"QQ Plot - {col}"
    )

    plt.show()

THis result strongly favors student-t Copula over, say, Gaussian Copula. Because A Student-t copula is specifically designed for Extreme events occurring together, which is exactly what we want in this integrated model. Gaussian copulas assume dependence generated from normally distributed latent variables. These Residuals are clearly heavy-tailed

Creating Pseudo-Observatons

In [ ]:
# converting residuals to ranks
# this is more robust to outliers and preserves observed tail behavior
from scipy.stats import rankdata

u = residuals.copy()

for col in u.columns:

    u[col] = (
        rankdata(
            u[col],
            method="average"
        )
        /
        (len(u) + 1)
    )

u.head()

In [ ]:
u.describe()

In [ ]:
print(
    u.corr(method="spearman")
)

Even with modest dependence, Copulas were employed to preserve any residual dependence structure among forecast errors during Monte Carlo simulation, ensuring that stochastic scenarios reflected observed joint behavior.

In [ ]:
# ==========================================================
# COPULA IMPORTS
# ==========================================================
from scipy.stats import norm
from scipy.stats import t
from scipy.stats import multivariate_normal
from scipy.optimize import minimize

from numpy.linalg import det, inv

In [ ]:
# ==========================================================
# GAUSSIAN COPULA
# ==========================================================

z_gauss = pd.DataFrame(
    norm.ppf(u),
    columns=u.columns,
    index=u.index
)

R_gauss = z_gauss.corr()

print("Gaussian Copula Correlation Matrix")
print()
print(R_gauss.round(4))

In [ ]:
# ==========================================================
# GAUSSIAN COPULA LOG-LIKELIHOOD
# ==========================================================

R = R_gauss.values

R_inv = inv(R)

R_det = det(R)

z = z_gauss.values

k = z.shape[1]

ll_gauss = 0

for i in range(len(z)):

    zi = z[i]

    ll_gauss += (
        -0.5*np.log(R_det)
        -0.5*zi @ (R_inv - np.eye(k)) @ zi
    )

print()
print("Gaussian Copula Log-Likelihood")
print(round(ll_gauss,4))

In [ ]:
# ==========================================================
# STUDENT-T COPULA
# ==========================================================

nu = 5

z_t = pd.DataFrame(
    t.ppf(u, df=nu),
    columns=u.columns,
    index=u.index
)

R_t = z_t.corr()

print("Student-t Copula Correlation Matrix")
print()
print(R_t.round(4))

In [ ]:
# ==========================================================
# STUDENT-T COPULA LOG-LIKELIHOOD
# ==========================================================

from scipy.special import gammaln

def t_copula_loglik(nu):

    z = t.ppf(u, df=nu)

    R = np.corrcoef(
        z.T
    )

    R_inv = np.linalg.inv(R)

    R_det = np.linalg.det(R)

    d = z.shape[1]

    ll = 0

    for i in range(len(z)):

        zi = z[i]

        term1 = (
            gammaln((nu+d)/2)
            - gammaln(nu/2)
            - (d/2)*np.log(nu*np.pi)
            - 0.5*np.log(R_det)
        )

        term2 = (
            -(nu+d)/2
            * np.log(
                1 +
                (zi @ R_inv @ zi)/nu
            )
        )

        term3 = np.sum(
            t.logpdf(
                zi,
                df=nu
            )
        )

        ll += (
            term1
            + term2
            - term3
        )

    return -ll

In [ ]:
# ==========================================================
# ESTIMATE BEST NU
# ==========================================================

opt = minimize(
    lambda x: t_copula_loglik(x[0]),
    x0=[5],
    bounds=[(2.1,50)]
)

best_nu = opt.x[0]

print()
print("Optimal Degrees of Freedom")
print(round(best_nu,4))

In [ ]:
# ==========================================================
# FINAL STUDENT-T LIKELIHOOD
# ==========================================================

ll_t = -opt.fun

print()
print("Student-t Copula Log-Likelihood")
print(round(ll_t,4))

In [ ]:
# ==========================================================
# COPULA COMPARISON
# ==========================================================

comparison = pd.DataFrame({
    "Copula":[
        "Gaussian",
        "Student-t"
    ],
    "LogLik":[
        ll_gauss,
        ll_t
    ]
})

print(comparison)

from the comparsion result above (LogLik compariosn and optimal degree of freedom), it's clear that Student-t copulas is the most effective to be used